# Module 10: Scientific Visualization with Matplotlib
## Spatial Rainfall Analysis of Ethiopian CHIRPS Data

This module covers scientific visualization techniques for spatial data: heatmaps, contour plots, filled contours, 3D surface plots, and quiver/stream plots using CHIRPS rainfall data.

## Setup

Load required libraries and datasets.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
from scipy import stats, ndimage
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mesh import continental_scale
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully!')

In [ ]:
DATA_DIR = '../datasets'
CHIRPS_PATH = '../chirps_1981_2022.nc'

ds = xr.open_dataset(CHIRPS_PATH, engine='netcdf4')
clim = xr.open_dataset(f'{DATA_DIR}/chirps_climatology.nc', engine='netcdf4')
annual = xr.open_dataset(f'{DATA_DIR}/chirps_annual.nc', engine='netcdf4')
eth = xr.open_dataset(f'{DATA_DIR}/chirps_ethiopia.nc', engine='netcdf4')

print(f'Full: {ds.precip.shape}, Clim: {clim.precip.shape}, Annual: {annual.precip.shape}, Eth: {eth.precip.shape}')

## 1. Heatmaps with pcolormesh

Heatmaps (pcolormesh) are ideal for visualizing spatial patterns in rainfall data. We'll plot mean annual rainfall across Ethiopia.

In [ ]:
mean_annual = annual.precip.mean(dim='time')

fig, ax = plt.subplots(figsize=(14, 8), subplot_kw={'projection': ccrs.PlateCarree()})

lon = mean_annual.longitude.values
lat = mean_annual.latitude.values

pcm = ax.pcolormesh(lon, lat, mean_annual.values,
                    cmap='Spectral_r', shading='auto',
                    vmin=0, vmax=1500)

ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.5, linestyle=':', alpha=0.7)
ax.add_feature(cfeature.LAKES, alpha=0.3, color='lightblue')
ax.add_feature(cfeature.RIVERS, linewidth=0.3, alpha=0.5)

ax.set_extent([30, 60, -5, 20], crs=ccrs.PlateCarree())

gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5, linestyle='--')
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {'size': 9}
gl.ylabel_style = {'size': 9}

cbar = plt.colorbar(pcm, ax=ax, orientation='horizontal', pad=0.05, aspect=40, shrink=0.8)
cbar.set_label('Mean Annual Precipitation (mm/year)', fontsize=11)

ax.set_title('Mean Annual Rainfall from CHIRPS (1981-2022)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

### Zoomed Heatmap: Ethiopia Region

Focused view on Ethiopia and the surrounding Horn of Africa region.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10), subplot_kw={'projection': ccrs.PlateCarree()})

eth_mean = eth.precip.mean(dim='time')

pcm = ax.pcolormesh(eth.longitude, eth.latitude, eth_mean.values,
                    cmap='viridis', shading='auto', vmin=0, vmax=250)

ax.add_feature(cfeature.COASTLINE, linewidth=1.0)
ax.add_feature(cfeature.BORDERS, linewidth=0.8, edgecolor='black')
ax.add_feature(cfeature.LAKES, alpha=0.4, color='lightblue', edgecolor='black', linewidth=0.5)
ax.add_feature(cfeature.RIVERS, linewidth=0.4, alpha=0.6)
ax.add_feature(cfeature.OCEAN, alpha=0.2, color='azure')

ax.set_extent([33, 48, 3, 15], crs=ccrs.PlateCarree())

gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5, linestyle='--')
gl.top_labels = False
gl.right_labels = False

cbar = plt.colorbar(pcm, ax=ax, orientation='vertical', pad=0.04, aspect=30, shrink=0.7)
cbar.set_label('Mean Monthly Precipitation (mm/month)', fontsize=11)

ax.set_title('Mean Monthly Rainfall - Ethiopia (CHIRPS 1981-2022)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 2. Contour Plots

Contour lines represent lines of equal rainfall amount (isohyets). Useful for visualizing spatial gradients.

In [ ]:
mean_annual_eth = eth.precip.mean(dim='time')
lon2d, lat2d = np.meshgrid(mean_annual_eth.longitude.values, mean_annual_eth.latitude.values)

fig, ax = plt.subplots(figsize=(12, 10), subplot_kw={'projection': ccrs.PlateCarree()})

contour_levels = np.arange(20, 261, 20)
cs = ax.contour(lon2d, lat2d, mean_annual_eth.values.T,
                levels=contour_levels, colors='darkblue', linewidths=1.2)

ax.clabel(cs, inline=True, fontsize=8, fmt='%d mm')

ax.add_feature(cfeature.COASTLINE, linewidth=1.0)
ax.add_feature(cfeature.BORDERS, linewidth=0.8)
ax.add_feature(cfeature.LAKES, alpha=0.3, color='lightblue')

ax.set_extent([33, 48, 3, 15], crs=ccrs.PlateCarree())

gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.4, linestyle='--')
gl.top_labels = False
gl.right_labels = False

ax.set_title('Mean Monthly Rainfall Contours - Ethiopia (mm/month)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Filled Contours (contourf)

Filled contours combine contour lines with color-filled regions for more impactful visualization.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10), subplot_kw={'projection': ccrs.PlateCarree()})

cf = ax.contourf(lon2d, lat2d, mean_annual_eth.values.T,
                 levels=np.arange(0, 281, 20), cmap='RdYlBu_r', extend='both')

cs = ax.contour(lon2d, lat2d, mean_annual_eth.values.T,
                levels=np.arange(40, 281, 40), colors='black', linewidths=0.8)
ax.clabel(cs, inline=True, fontsize=7, fmt='%d')

ax.add_feature(cfeature.COASTLINE, linewidth=1.0)
ax.add_feature(cfeature.BORDERS, linewidth=0.8)
ax.add_feature(cfeature.LAKES, alpha=0.4, color='lightblue', edgecolor='black', linewidth=0.5)

ax.set_extent([33, 48, 3, 15], crs=ccrs.PlateCarree())

gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.4, linestyle='--')
gl.top_labels = False
gl.right_labels = False

cbar = plt.colorbar(cf, ax=ax, orientation='horizontal', pad=0.05, aspect=40, shrink=0.8)
cbar.set_label('Mean Monthly Precipitation (mm/month)', fontsize=11)

ax.set_title('Filled Contour Map of Mean Monthly Rainfall - Ethiopia', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 4. 3D Surface Plots

Surface plots show rainfall as a 3D topographic surface, revealing spatial patterns dramatically.

In [ ]:
rain_2d = mean_annual_eth.values.T  # (lat, lon)

fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection='3d')

X, Y = np.meshgrid(mean_annual_eth.longitude.values, mean_annual_eth.latitude.values)
Z = rain_2d

surf = ax.plot_surface(X, Y, Z, cmap='terrain',
                       linewidth=0, antialiased=True,
                       rstride=2, cstride=2)

ax.contour(X, Y, Z, zdir='z', offset=-20, cmap='terrain', linewidths=0.5)

ax.set_xlabel('Longitude', fontsize=11, labelpad=10)
ax.set_ylabel('Latitude', fontsize=11, labelpad=10)
ax.set_zlabel('Precipitation (mm/month)', fontsize=11, labelpad=10)
ax.set_title('3D Surface of Mean Monthly Rainfall - Ethiopia', fontsize=14, fontweight='bold')

cbar = fig.colorbar(surf, ax=ax, orientation='horizontal', shrink=0.6, pad=0.1)
cbar.set_label('mm/month', fontsize=11)

ax.view_init(elev=25, azim=135)

plt.tight_layout()
plt.show()

### 3D Wireframe Plot

An alternative 3D representation using wireframe.

In [ ]:
fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection='3d')

wframe = ax.plot_wireframe(X[::3, ::3], Y[::3, ::3], Z[::3, ::3],
                           rstride=3, cstride=3, color='steelblue',
                           linewidth=0.5, alpha=0.7)

ax.set_xlabel('Longitude', fontsize=11, labelpad=10)
ax.set_ylabel('Latitude', fontsize=11, labelpad=10)
ax.set_zlabel('Precipitation (mm/month)', fontsize=11, labelpad=10)
ax.set_title('3D Wireframe of Mean Monthly Rainfall - Ethiopia', fontsize=14, fontweight='bold')

ax.view_init(elev=30, azim=120)

plt.tight_layout()
plt.show()

## 5. Flow/Vector Visualization

We simulate wind-like flow from rainfall gradients using quiver plots. This shows the direction of maximum rainfall increase (gradient vectors).

In [ ]:
rain_smooth = ndimage.gaussian_filter(rain_2d, sigma=2.0)
dy, dx = np.gradient(rain_smooth)

skip = 8
x_sub = X[::skip, ::skip]
y_sub = Y[::skip, ::skip]
u_sub = dx[::skip, ::skip]
v_sub = dy[::skip, ::skip]

magnitude = np.sqrt(u_sub**2 + v_sub**2)

fig, ax = plt.subplots(figsize=(12, 10), subplot_kw={'projection': ccrs.PlateCarree()})

cf = ax.contourf(X, Y, rain_2d, levels=15, cmap='YlGnBu', alpha=0.7)

q = ax.quiver(x_sub, y_sub, u_sub, v_sub, magnitude,
              cmap='plasma', scale=500, width=0.003,
              headwidth=4, headlength=5, alpha=0.8)

ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.add_feature(cfeature.LAKES, alpha=0.3, color='lightblue')

ax.set_extent([33, 48, 3, 15], crs=ccrs.PlateCarree())

gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.4, linestyle='--')
gl.top_labels = False
gl.right_labels = False

cbar = plt.colorbar(cf, ax=ax, orientation='horizontal', pad=0.05, aspect=40, shrink=0.8)
cbar.set_label('Precipitation (mm/month)', fontsize=11)

ax.set_title('Rainfall Gradient Vectors over Ethiopia', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Multi-Panel Seasonal Maps

Create a 2×2 grid of filled contour maps for each season using cartopy.

In [ ]:
season_months = {
    'DJF (Dry)': [12, 1, 2],
    'MAM (Long Rain)': [3, 4, 5],
    'JJA (Main Rain)': [6, 7, 8],
    'SON (Short Rain)': [9, 10, 11]
}

seasonal_means = {}
for name, months in season_months.items():
    mask = eth.time.dt.month.isin(months)
    seasonal_means[name] = eth.precip.where(mask).mean(dim='time')

fig, axes = plt.subplots(2, 2, figsize=(16, 12),
                         subplot_kw={'projection': ccrs.PlateCarree()})
axes = axes.flatten()

for idx, (name, data) in enumerate(seasonal_means.items()):
    ax = axes[idx]

    cf = ax.contourf(data.longitude, data.latitude, data.values.T,
                     levels=np.arange(0, 351, 25), cmap='RdYlBu_r', extend='max')

    ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.add_feature(cfeature.LAKES, alpha=0.3, color='lightblue')

    ax.set_extent([33, 48, 3, 15], crs=ccrs.PlateCarree())

    gl = ax.gridlines(draw_labels=True, linewidth=0.2, alpha=0.3, linestyle='--')
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {'size': 8}
    gl.ylabel_style = {'size': 8}

    ax.set_title(name, fontsize=12, fontweight='bold')

cbar = fig.colorbar(cf, ax=axes, orientation='horizontal', pad=0.05, aspect=50, shrink=0.6)
cbar.set_label('Precipitation (mm/month)', fontsize=11)

fig.suptitle('Seasonal Rainfall Patterns over Ethiopia', fontsize=16, fontweight='bold', y=1.02)

plt.tight_layout()
plt.show()

### Difference Map: Wet Season vs Dry Season

Highlight the contrast between the main rainy season (JJA) and the dry season (DJF).

In [ ]:
diff = seasonal_means['JJA (Main Rain)'] - seasonal_means['DJF (Dry)']

fig, ax = plt.subplots(figsize=(12, 10), subplot_kw={'projection': ccrs.PlateCarree()})

vmax = max(abs(diff.values).max(), 1)
cf = ax.contourf(diff.longitude, diff.latitude, diff.values.T,
                 levels=np.linspace(-vmax, vmax, 21),
                 cmap='RdBu_r', extend='both')

ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.add_feature(cfeature.LAKES, alpha=0.3, color='lightblue')
ax.set_extent([33, 48, 3, 15], crs=ccrs.PlateCarree())

gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.4, linestyle='--')
gl.top_labels = False
gl.right_labels = False

cbar = plt.colorbar(cf, ax=ax, orientation='horizontal', pad=0.05, aspect=40, shrink=0.8)
cbar.set_label('Difference (mm/month)', fontsize=11)

ax.set_title('JJA minus DJF: Wet-Dry Season Rainfall Difference', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## Exercises

1. **High-resolution Heatmap**: Create a pcolormesh of July rainfall over Ethiopia with a terrain colormap and custom colorbar ticks.
2. **Contour Animation**: (Conceptual) Describe how you would animate contour maps for each month of the year.
3. **3D Surface Comparison**: Create side-by-side 3D surfaces for the wettest and driest months (July and January).
4. **Seasonal Averages**: Create a 1×3 panel of filled contours for the Long Rain (MAM), Main Rain (JJA), and Short Rain (SON) seasons.
5. **Stream Plot**: Using the gradient approach, create a stream plot overlay on a filled contour map of the Ethiopian highlands.

### Mini-Project: Scientific Rainfall Atlas of Ethiopia

Create a publication-ready scientific figure with:
- A main panel showing mean annual rainfall with contour lines over Ethiopia (cartopy)
- Three inset panels showing seasonal maps (DJF, MAM, JJA, SON) arranged in a 2×2 grid
- A difference panel showing the anomaly of the wettest year vs the long-term mean
- All panels with proper colorbars, coordinate grids, coastlines, and borders
- Export the figure as a high-resolution PNG (300 dpi)